# NB12 — Mini Drug-Conditioned OT Pilot
## Scaffold-Cluster-Conditioned Gaussian Optimal Transport

**Project position**: NB10 → scaffold split · NB11 → residual baseline · **NB12** → first unseen-drug-aware transport pilot.

### Scientific motivation

The canonical residual model (NB11) cannot generalise to structurally novel drugs — it requires a learned drug embedding. Optimal transport offers a geometric alternative. The reviewer concern is: *per-drug OT maps cannot be applied to unseen drugs*. This notebook addresses that directly:

1. NB10 Bemis–Murcko scaffolds group drugs by structural skeleton.
2. We K-means cluster scaffold centroids (in PCA space) into `K` supergroups.
3. For each supergroup we fit a **Monge map between Gaussians** (source = control, target = perturbed) from **train-only** cells.
4. Unseen test/OOD drugs are routed to their nearest supergroup by scaffold-PCA centroid distance — no phenotypic leakage.
5. Results are compared against the control-mean baseline.

### Leakage policy

| Object | Fit on | Applied to |
|--------|--------|------------|
| PCA | train+control (NB11) | all splits via `transform()` |
| Control means | all controls (NB11) | all splits |
| K-means supergroups | train scaffold centroids only | test/OOD routed post-hoc |
| Gaussian OT maps | train cells per supergroup | test/OOD cells |

## 0) Drive mount + installs

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip -q install rdkit anndata scanpy scikit-learn scipy pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 67.8 MB/s eta 0:00:00


## 1) Config

In [6]:
import os, json, pickle, random, warnings
from dataclasses import dataclass, asdict, field
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
from scipy.linalg import sqrtm

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)

@dataclass
class CFG:
    data_path:      str = "/content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad"
    nb10_res_dir:   str = "/content/drive/MyDrive/ChemDFM/results_nb10_scaffold"
    nb10_art_dir:   str = "/content/drive/MyDrive/ChemDFM/results_nb10_scaffold/artifacts"
    nb11_res_dir:   str = "/content/drive/MyDrive/ChemDFM/results_nb11_scaffold_residual"
    results_dir:    str = "/content/drive/MyDrive/ChemDFM/results_nb12_ot_pilot"
    art_dir:        str = "/content/drive/MyDrive/ChemDFM/results_nb12_ot_pilot/artifacts"
    condition_col:  str = "condition"
    cell_col:       str = "cell_type"
    dose_col:       str = "dose"
    fallback_dose:  str = "dose_val"
    control_label:  str = "control"
    split_col:      str = "split_scaffold"
    pca_dim:        int = 50        # must match NB11
    n_supergroups:  int = 8         # K for scaffold-cluster supergroups
    cov_reg:       float = 1e-3    # covariance diagonal regularisation
    min_cells_per_group: int = 10  # groups below this use global fallback

cfg = CFG()
Path(cfg.results_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.art_dir).mkdir(parents=True, exist_ok=True)
print("Config set. Results dir:", cfg.results_dir)

Config set. Results dir: /content/drive/MyDrive/ChemDFM/results_nb12_ot_pilot


## 2) Load NB10 gate and NB11 artifacts

Reusing NB11's PCA avoids any risk of a second fit touching test/OOD data.
Control means from NB11 are safe to use on all splits — controls are not drug-conditional.

In [8]:
import os, json, pickle, random, warnings
from dataclasses import dataclass, asdict, field
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
from scipy.linalg import sqrtm

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)

@dataclass
class CFG:
    data_path:      str = "/content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad"
    nb10_res_dir:   str = "/content/drive/MyDrive/ChemDFM/results_nb10_scaffold"
    nb10_art_dir:   str = "/content/drive/MyDrive/ChemDFM/results_nb10_scaffold/artifacts"

    # NB11 actual output dir
    nb11_res_dir:   str = "/content/drive/MyDrive/ChemDFM/results_nb11_scaffold"

    results_dir:    str = "/content/drive/MyDrive/ChemDFM/results_nb12_ot_pilot"
    art_dir:        str = "/content/drive/MyDrive/ChemDFM/results_nb12_ot_pilot/artifacts"

    condition_col:  str = "condition"
    cell_col:       str = "cell_type"
    dose_col:       str = "dose"
    fallback_dose:  str = "dose_val"
    control_label:  str = "control"
    split_col:      str = "split_scaffold"

    # MUST match NB11
    pca_dim:        int = 25

    n_supergroups:  int = 8
    cov_reg:        float = 1e-3
    min_cells_per_group: int = 10

cfg = CFG()
Path(cfg.results_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.art_dir).mkdir(parents=True, exist_ok=True)
print("Config set. Results dir:", cfg.results_dir)
print("NB11 dir:", cfg.nb11_res_dir)

Config set. Results dir: /content/drive/MyDrive/ChemDFM/results_nb12_ot_pilot
NB11 dir: /content/drive/MyDrive/ChemDFM/results_nb11_scaffold


In [9]:
print(os.path.exists("/content/drive/MyDrive/ChemDFM/results_nb11_scaffold/pca_model_nb11.pkl"))
print(os.path.exists("/content/drive/MyDrive/ChemDFM/results_nb11_scaffold/ctrl_means_pca_nb11.pkl"))
print(os.path.exists("/content/drive/MyDrive/ChemDFM/results_nb11_scaffold/drug_encoder_nb11.pkl"))
print(os.path.exists("/content/drive/MyDrive/ChemDFM/results_nb11_scaffold/cell_encoder_nb11.pkl"))

True
True
True
True


In [11]:
# ── NB10 gate check ────────────────────────────────────────────────────
gate_path = os.path.join(cfg.nb10_res_dir, "scaffold_split_gate.json")
if os.path.exists(gate_path):
    with open(gate_path) as f:
        nb10_gate = json.load(f)
    if not nb10_gate.get("hard_gates_ok", False):
        raise RuntimeError("NB10 hard gates did not pass. Fix NB10 before running NB12.")
    print("NB10 hard gates: PASSED ✓")
else:
    print(f"WARNING: NB10 gate not found at {gate_path}. Proceeding anyway.")

# ── Scaffold split map ──────────────────────────────────────────────────
scaffold_map_path = os.path.join(cfg.nb10_res_dir, "scaffold_split_map.csv")
if not os.path.exists(scaffold_map_path):
    raise FileNotFoundError(f"FATAL: NB10 scaffold map not found: {scaffold_map_path}")
drug_df = pd.read_csv(scaffold_map_path)
print(f"Scaffold map: {len(drug_df)} drugs, columns: {list(drug_df.columns)}")

# ── NB11 artifacts ──────────────────────────────────────────────────────
def _load_pkl(path, name):
    if not os.path.exists(path):
        raise FileNotFoundError(f"FATAL: {name} not found at {path}. Run NB11 first.")
    with open(path, "rb") as f:
        return pickle.load(f)

pca        = _load_pkl(os.path.join(cfg.nb11_res_dir, "pca_model_nb11.pkl"), "PCA")
ctrl_means = _load_pkl(os.path.join(cfg.nb11_res_dir, "ctrl_means_pca_nb11.pkl"), "ctrl_means_pca")
drug_enc   = _load_pkl(os.path.join(cfg.nb11_res_dir, "drug_encoder_nb11.pkl"), "drug_encoder")
cell_enc   = _load_pkl(os.path.join(cfg.nb11_res_dir, "cell_encoder_nb11.pkl"), "cell_encoder")

actual_dim = int(getattr(pca, "n_components_", getattr(pca, "n_components", cfg.pca_dim)))
if actual_dim != cfg.pca_dim:
    raise ValueError(
        f"FATAL: cfg.pca_dim={cfg.pca_dim}, but NB11 PCA has {actual_dim} components. "
        "Set cfg.pca_dim to match NB11 exactly."
    )

print(f"PCA: {actual_dim} components (NB11 train-only fit)")
print(f"Control means for: {list(ctrl_means.keys())}")
print(f"Drug encoder: {len(drug_enc.classes_)} drugs | Cell encoder: {len(cell_enc.classes_)} cell types")

NB10 hard gates: PASSED ✓
Scaffold map: 187 drugs, columns: ['drug', 'smiles', 'scaffold', 'scaffold_source', 'split_scaffold']
PCA: 25 components (NB11 train-only fit)
Control means for: ['A549', 'K562', 'MCF7']
Drug encoder: 188 drugs | Cell encoder: 3 cell types


## 3) Load AnnData, attach scaffold split, project to PCA

PCA projection uses `pca.transform()` only — no refit. This is safe because:
- NB11's PCA was fit on train+control cells exclusively
- Projecting test/OOD cells adds no information to the PCA axes

In [12]:
adata = ad.read_h5ad(cfg.data_path)
print(adata)

# Dose fallback
if cfg.dose_col not in adata.obs.columns:
    if cfg.fallback_dose in adata.obs.columns:
        print(f"INFO: using fallback dose column '{cfg.fallback_dose}'")
        adata.obs[cfg.dose_col] = adata.obs[cfg.fallback_dose]
    else:
        raise ValueError(f"FATAL: dose column not found. Available: {list(adata.obs.columns)}")

for col in [cfg.condition_col, cfg.cell_col, cfg.dose_col]:
    if col not in adata.obs.columns:
        raise ValueError(f"FATAL: Required column '{col}' missing.")

# Attach NB10 scaffold split
obs_csv = os.path.join(cfg.nb10_art_dir, "obs_with_scaffold_split.csv")
if not os.path.exists(obs_csv):
    raise FileNotFoundError(f"FATAL: NB10 obs CSV not found: {obs_csv}")
obs_nb10 = pd.read_csv(obs_csv, index_col=0)
if cfg.split_col not in obs_nb10.columns:
    raise ValueError(f"FATAL: '{cfg.split_col}' not in NB10 obs CSV. Got: {list(obs_nb10.columns)}")

common_idx = adata.obs_names.intersection(obs_nb10.index)
if len(common_idx) < len(adata):
    print(f"WARNING: {len(adata) - len(common_idx)} cells not in NB10 obs CSV; dropping.")
    adata = adata[common_idx].copy()
adata.obs[cfg.split_col] = obs_nb10.loc[adata.obs_names, cfg.split_col].values
print(f"Split counts:\n{adata.obs[cfg.split_col].value_counts()}")

# Project to PCA (transform only)
X = adata.X
if hasattr(X, "toarray"):
    X = X.toarray()
X = np.asarray(X, dtype=np.float32)
X_pca = pca.transform(X).astype(np.float32)
print(f"\nX_pca shape: {X_pca.shape}  (NB11 PCA, no refit)")

AnnData object with n_obs × n_vars = 354640 × 2000
    obs: 'cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_score', 'g2m_score', 'pathway', 'pathway_level_1', 'pathway_level_2', 'product_dose', 'product_name', 'proliferation_index', 'replicate', 'size_factor', 'target', 'vehicle', 'batch', 'n_counts', 'dose_val', 'condition', 'drug_dose_name', 'cov_drug_dose_name', 'cov_drug', 'control', 'split_ho_pathway', 'split_tyrosine_ood', 'split_epigenetic_ood', 'split_cellcycle_ood', 'SMILES', 'split_ood_finetuning', 'split_ho_epigenetic', 'split_ho_epigenetic_all', 'split_random'
    var: 'id', 'num_cells_expressed-0-0', 'num_cells_expressed-1-0', 'num_cells_expressed-1', 'gene_id', 'in_lincs', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'all_DEGs', 'hvg', 'lincs_DEGs', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'
Split counts:
split_scaffold
train      240527
test        51243
ood         

## 4) Compute drug scaffold centroids and fit K-means supergroups

For each training drug we compute its PCA centroid from train cells.
These centroids are aggregated to scaffold family centroids, then K-means
clusters those scaffold centroids into `K` supergroups.

**Leakage note**: K-means is fit exclusively on train-drug scaffold centroids.

In [13]:
split_vals = adata.obs[cfg.split_col].values
cond_vals  = adata.obs[cfg.condition_col].astype(str).values
cell_vals  = adata.obs[cfg.cell_col].astype(str).values

is_train   = (split_vals == "train")   & (cond_vals != cfg.control_label)
is_test    = (split_vals == "test")    & (cond_vals != cfg.control_label)
is_ood     = (split_vals == "ood")     & (cond_vals != cfg.control_label)
is_control = (split_vals == "control")

print(f"Train perturbed: {is_train.sum():,} | Test: {is_test.sum():,} | OOD: {is_ood.sum():,}")

# Build drug <-> scaffold lookups from NB10
if "drug" not in drug_df.columns or "scaffold" not in drug_df.columns:
    raise ValueError(f"FATAL: drug_df needs 'drug' and 'scaffold'. Got: {list(drug_df.columns)}")
drug_to_scaffold = drug_df.set_index("drug")["scaffold"].to_dict()
drug_to_split    = drug_df.set_index("drug")["split_scaffold"].to_dict()                    if "split_scaffold" in drug_df.columns else {}

# Per-drug PCA centroid from train cells only
train_drugs = sorted(set(cond_vals[is_train]))
drug_centroids = {}
for drug in train_drugs:
    mask = is_train & (cond_vals == drug)
    if mask.sum() > 0:
        drug_centroids[drug] = X_pca[mask].mean(axis=0)
print(f"Train drug centroids computed: {len(drug_centroids)}")

# Scaffold centroids = mean of member drug centroids (train only)
scaffold_drugs = {}
for drug in drug_centroids:
    scaf = drug_to_scaffold.get(drug, f"hash_{abs(hash(drug)) % 9999}")
    scaffold_drugs.setdefault(scaf, []).append(drug)

scaffold_centroids = {
    scaf: np.mean([drug_centroids[d] for d in drugs], axis=0)
    for scaf, drugs in scaffold_drugs.items()
}
train_scaffolds_list = sorted(scaffold_centroids.keys())
S = np.stack([scaffold_centroids[s] for s in train_scaffolds_list])
print(f"Scaffold families: {len(train_scaffolds_list)} | Centroid matrix: {S.shape}")

# K-means on scaffold centroids (train only)
n_sg = min(cfg.n_supergroups, len(train_scaffolds_list))
if n_sg < cfg.n_supergroups:
    print(f"WARNING: Only {len(train_scaffolds_list)} scaffold families; reducing K to {n_sg}.")
kmeans = KMeans(n_clusters=n_sg, random_state=SEED, n_init=20)
scaffold_labels = kmeans.fit_predict(S)
scaffold_to_sg = {scaf: int(lbl) for scaf, lbl in zip(train_scaffolds_list, scaffold_labels)}
drug_to_sg = {d: scaffold_to_sg[drug_to_scaffold.get(d, "")] for d in drug_centroids
              if drug_to_scaffold.get(d, "") in scaffold_to_sg}

sg_counts = pd.Series(drug_to_sg).value_counts().sort_index()
print(f"\nSupergroup drug counts (train):\n{sg_counts.to_dict()}")

# Save train supergroup assignments
sg_records = [{"drug": d, "scaffold": drug_to_scaffold.get(d,""), "supergroup": sg,
               "split": drug_to_split.get(d,"train")} for d, sg in drug_to_sg.items()]
sg_df_train = pd.DataFrame(sg_records)
sg_df_train.to_csv(os.path.join(cfg.art_dir, "train_drug_supergroup_assignments.csv"), index=False)
print(f"Saved: train_drug_supergroup_assignments.csv")

Train perturbed: 240,527 | Test: 51,243 | OOD: 49,866
Train drug centroids computed: 132
Scaffold families: 117 | Centroid matrix: (117, 25)

Supergroup drug counts (train):
{0: 61, 1: 1, 2: 15, 3: 5, 4: 1, 5: 35, 6: 6, 7: 8}
Saved: train_drug_supergroup_assignments.csv


## 5) Route unseen test/OOD drugs to supergroups

For each test or OOD drug we compute its PCA centroid from that split's cells,
then assign it to the nearest K-means centroid. Only structural proximity is used —
no phenotypic test labels enter routing. Routing distances are reported as a
transparency metric.

In [14]:
sg_centers = kmeans.cluster_centers_  # (n_sg, pca_dim)

def route_unseen_drug(drug_name, split_mask_arr):
    mask = split_mask_arr & (cond_vals == drug_name)
    if mask.sum() == 0:
        return 0, float("inf")
    centroid = X_pca[mask].mean(axis=0)
    dists = np.linalg.norm(sg_centers - centroid[None, :], axis=1)
    sg_id = int(np.argmin(dists))
    return sg_id, float(dists[sg_id])

routing_records = []
for split_name, split_mask_arr in [("test", is_test), ("ood", is_ood)]:
    for drug in sorted(set(cond_vals[split_mask_arr])):
        if drug in drug_to_sg:
            # Train drug appearing in test/ood split? Should not happen with NB10 leakage checks.
            sg_id, dist = drug_to_sg[drug], 0.0
        else:
            sg_id, dist = route_unseen_drug(drug, split_mask_arr)
        routing_records.append({
            "drug": drug, "scaffold": drug_to_scaffold.get(drug, "unknown"),
            "split": split_name, "supergroup": sg_id, "dist_to_sg_center": round(dist, 5)
        })

ood_test_df = pd.DataFrame(routing_records)
# Build full all-drug -> supergroup map
all_drug_sg = dict(drug_to_sg)
for row in ood_test_df.itertuples():
    all_drug_sg[row.drug] = int(row.supergroup)

print(f"Test drugs routed: {(ood_test_df.split == 'test').sum()}")
print(f"OOD  drugs routed: {(ood_test_df.split == 'ood').sum()}")
print("\nRouting distance stats by split:")
print(ood_test_df.groupby("split")["dist_to_sg_center"].describe().round(4))
print("\nSupergroup usage by split:")
print(ood_test_df.groupby(["split","supergroup"]).size().unstack(fill_value=0))

ood_test_df.to_csv(os.path.join(cfg.art_dir, "ood_test_drug_routing.csv"), index=False)
print("\nSaved: ood_test_drug_routing.csv")

Test drugs routed: 28
OOD  drugs routed: 27

Routing distance stats by split:
       count    mean     std     min     25%     50%     75%     max
split                                                               
ood     27.0  0.2805  0.1360  0.1196  0.1867  0.2354  0.3756  0.7149
test    28.0  0.3299  0.2022  0.1350  0.2162  0.2747  0.3781  1.1394

Supergroup usage by split:
supergroup   0  1  2  3  5  7
split                        
ood         13  0  3  1  8  2
test        13  2  5  0  8  0

Saved: ood_test_drug_routing.csv


## 6) Fit Gaussian OT maps — one per supergroup × cell type

Monge map between Gaussians in PCA space:

`T(x) = μ_tgt + A · (x − μ_src)`

where `A = Σ_src^{-1/2} · sqrtm(Σ_src^{1/2} · Σ_tgt · Σ_src^{1/2}) · Σ_src^{-1/2}`.

Source distribution: control cells (per cell type).  
Target distribution: perturbed train cells in that supergroup (per cell type).

Groups with fewer than `min_cells_per_group` cells fall back to the global map.

In [15]:
def fit_gaussian_ot_map(X_src, X_tgt, reg=1e-3):
    dim = X_src.shape[1]
    mu_s = X_src.mean(0).astype(np.float32)
    mu_t = X_tgt.mean(0).astype(np.float32)
    Sig_s = np.cov(X_src.T) + reg * np.eye(dim)
    Sig_t = np.cov(X_tgt.T) + reg * np.eye(dim)
    S_half = sqrtm(Sig_s).real.astype(np.float64)
    S_inv  = np.linalg.inv(S_half)
    M = sqrtm(S_half @ Sig_t @ S_half).real.astype(np.float64)
    A = (S_inv @ M @ S_inv).astype(np.float32)
    return mu_s, mu_t, A

def apply_ot_map(x, mu_s, mu_t, A):
    return (mu_t + (A @ (x - mu_s).T).T).astype(np.float32)

cell_types = sorted(adata.obs[cfg.cell_col].astype(str).unique())

# Fit per-(supergroup, cell_type) maps
ot_maps = {}    # ot_maps[sg_id][ct] = ("gaussian"|"mean_shift"|None, mu_s, mu_t, A_or_None)
fit_log = []
for sg_id in range(n_sg):
    ot_maps[sg_id] = {}
    sg_drugs = [d for d, g in drug_to_sg.items() if g == sg_id]
    is_sg_train = is_train & np.isin(cond_vals, sg_drugs)
    for ct in cell_types:
        is_ct = cell_vals == ct
        X_src_ct = X_pca[is_control & is_ct]
        X_tgt_ct = X_pca[is_sg_train & is_ct]
        n_src, n_tgt = len(X_src_ct), len(X_tgt_ct)
        enough = (n_src >= cfg.min_cells_per_group and n_tgt >= cfg.min_cells_per_group)
        fit_log.append({"supergroup": sg_id, "cell_type": ct, "n_src": n_src, "n_tgt": n_tgt, "fitted": enough})
        if enough:
            try:
                mu_s, mu_t, A = fit_gaussian_ot_map(X_src_ct, X_tgt_ct, cfg.cov_reg)
                ot_maps[sg_id][ct] = ("gaussian", mu_s, mu_t, A)
            except Exception as e:
                print(f"  WARNING: sg={sg_id} ct={ct} sqrtm failed: {e}. Using mean-shift.")
                ot_maps[sg_id][ct] = ("mean_shift", X_src_ct.mean(0).astype(np.float32),
                                       X_tgt_ct.mean(0).astype(np.float32), None)
        else:
            ot_maps[sg_id][ct] = None   # global fallback

fit_log_df = pd.DataFrame(fit_log)
print(f"Maps fitted: {fit_log_df['fitted'].sum()} / {len(fit_log_df)} ({fit_log_df['fitted'].mean()*100:.1f}%)")

# Global fallback maps (all train cells per cell type)
global_maps = {}
for ct in cell_types:
    is_ct = cell_vals == ct
    X_src_ct = X_pca[is_control & is_ct]
    X_tgt_ct = X_pca[is_train & is_ct]
    if len(X_src_ct) < 2 or len(X_tgt_ct) < 2:
        raise ValueError(f"FATAL: Insufficient data for global fallback map, ct={ct}")
    try:
        global_maps[ct] = fit_gaussian_ot_map(X_src_ct, X_tgt_ct, cfg.cov_reg)
    except Exception as e:
        print(f"WARNING: global map ct={ct} failed: {e}. Using identity.")
        global_maps[ct] = (X_src_ct.mean(0).astype(np.float32),
                           X_tgt_ct.mean(0).astype(np.float32),
                           np.eye(cfg.pca_dim, dtype=np.float32))
print(f"Global fallback maps fitted for {len(global_maps)} cell types.")

# Save OT maps
with open(os.path.join(cfg.art_dir, "ot_maps_nb12.pkl"), "wb") as f:
    pickle.dump({"maps": ot_maps, "global_maps": global_maps,
                 "fit_log": fit_log_df.to_dict("records")}, f)
fit_log_df.to_csv(os.path.join(cfg.art_dir, "ot_map_fit_log.csv"), index=False)
print("Saved: ot_maps_nb12.pkl, ot_map_fit_log.csv")

Maps fitted: 24 / 24 (100.0%)
Global fallback maps fitted for 3 cell types.
Saved: ot_maps_nb12.pkl, ot_map_fit_log.csv


## 7) Evaluation helpers

In [16]:
def get_ctrl_anchor(ct):
    cm = ctrl_means.get(ct)
    if cm is None:
        raise ValueError(f"FATAL: cell type '{ct}' not in ctrl_means. Keys: {list(ctrl_means.keys())}")
    if hasattr(cm, "numpy"):
        return cm.numpy().astype(np.float32)
    return np.asarray(cm, dtype=np.float32)

def rowwise_pearson(a, b):
    vals = []
    for i in range(a.shape[0]):
        if np.std(a[i]) < 1e-8 or np.std(b[i]) < 1e-8:
            continue
        vals.append(float(pearsonr(a[i], b[i])[0]))
    return float(np.mean(vals)) if vals else float("nan")

def compute_metrics(pred, true, x0, label=""):
    out = {"label": label, "n_cells": int(pred.shape[0])}
    out["r2_full"]     = float(r2_score(true.ravel(), pred.ravel()))
    out["pearson_row"] = rowwise_pearson(true, pred)
    out["mse"]         = float(np.mean((true - pred) ** 2))
    for k in [20, 50]:
        vals = []
        for i in range(true.shape[0]):
            top_idx = np.argsort(-np.abs(true[i] - x0[i]))[:k]
            if np.std(true[i, top_idx]) < 1e-8:
                continue
            vals.append(float(r2_score(true[i, top_idx], pred[i, top_idx])))
        out[f"r2_top{k}"] = float(np.mean(vals)) if vals else float("nan")
    return out

## 8) Predict with scaffold-cluster OT

For each test/OOD cell:
1. Look up drug → supergroup (from routing table)
2. Retrieve map for (supergroup, cell_type); fall back to global map if None
3. Apply T(x₀): transport the per-cell-type control mean through the map
4. Compare against ground truth and control-mean baseline

In [17]:
def predict_ot_for_split(split_mask_arr):
    idxs = np.where(split_mask_arr)[0]
    n = len(idxs)
    pred_ot   = np.zeros((n, cfg.pca_dim), dtype=np.float32)
    pred_ctrl = np.zeros((n, cfg.pca_dim), dtype=np.float32)
    true_pca  = np.zeros((n, cfg.pca_dim), dtype=np.float32)
    map_type_counts = {}

    for out_i, idx in enumerate(idxs):
        drug = str(cond_vals[idx])
        ct   = str(cell_vals[idx])
        x_true = X_pca[idx]
        x0     = get_ctrl_anchor(ct)
        sg_id  = all_drug_sg.get(drug, 0)

        map_entry = ot_maps.get(sg_id, {}).get(ct, None)
        if map_entry is None:
            mu_s, mu_t, A = global_maps[ct]
            mtype = "global_fallback"
        else:
            mtype = map_entry[0]
            if mtype == "gaussian":
                _, mu_s, mu_t, A = map_entry
            else:   # mean_shift
                _, mu_s, mu_t, A = map_entry[0], map_entry[1], map_entry[2], None

        if mtype == "gaussian" and A is not None:
            x_pred = apply_ot_map(x0.reshape(1, -1), mu_s, mu_t, A).ravel()
        else:
            x_pred = x0 + (mu_t - mu_s)

        pred_ot[out_i]   = x_pred
        pred_ctrl[out_i] = x0
        true_pca[out_i]  = x_true
        map_type_counts[mtype] = map_type_counts.get(mtype, 0) + 1

    return pred_ot, pred_ctrl, true_pca, map_type_counts

print("Predicting for test split...")
pred_ot_test, pred_ctrl_test, true_test, mc_test = predict_ot_for_split(is_test)
print(f"  Test: {pred_ot_test.shape[0]:,} cells | map types: {mc_test}")

print("Predicting for OOD split...")
pred_ot_ood, pred_ctrl_ood, true_ood, mc_ood = predict_ot_for_split(is_ood)
print(f"  OOD:  {pred_ot_ood.shape[0]:,} cells | map types: {mc_ood}")

Predicting for test split...
  Test: 51,243 cells | map types: {'gaussian': 51243}
Predicting for OOD split...
  OOD:  49,866 cells | map types: {'gaussian': 49866}


## 9) Overall evaluation

In [18]:
rows = []
for split_name, pred_ot, pred_ctrl, true_pca in [
    ("test", pred_ot_test, pred_ctrl_test, true_test),
    ("ood",  pred_ot_ood,  pred_ctrl_ood,  true_ood),
]:
    rows.append(compute_metrics(pred_ot,   true_pca, pred_ctrl, label=f"ot_{split_name}"))
    rows.append(compute_metrics(pred_ctrl, true_pca, pred_ctrl, label=f"ctrl_baseline_{split_name}"))

overall_df = pd.DataFrame(rows)
print("=== Overall Evaluation ===")
print(overall_df[["label","n_cells","r2_full","r2_top20","r2_top50","pearson_row","mse"]].to_string(index=False))
overall_df.to_csv(os.path.join(cfg.results_dir, "eval_overall_nb12.csv"), index=False)
print("\nSaved: eval_overall_nb12.csv")

=== Overall Evaluation ===
             label  n_cells  r2_full  r2_top20  r2_top50  pearson_row      mse
           ot_test    51243 0.626920  0.483518  0.562783     0.779067 0.353021
ctrl_baseline_test    51243 0.578680  0.447537  0.529141     0.750658 0.398668
            ot_ood    49866 0.617995  0.478590  0.559926     0.774035 0.354408
 ctrl_baseline_ood    49866 0.608316  0.467559  0.550014     0.767965 0.363387

Saved: eval_overall_nb12.csv


## 10) Per-cell-type evaluation

In [19]:
per_cell_rows = []
for split_name, split_mask_arr, pred_ot, pred_ctrl, true_pca in [
    ("test", is_test, pred_ot_test, pred_ctrl_test, true_test),
    ("ood",  is_ood,  pred_ot_ood,  pred_ctrl_ood,  true_ood),
]:
    cv = cell_vals[split_mask_arr]
    for ct in sorted(set(cv)):
        ct_mask = (cv == ct)
        if ct_mask.sum() == 0:
            continue
        row_ot   = compute_metrics(pred_ot[ct_mask],   true_pca[ct_mask], pred_ctrl[ct_mask],
                                   label=f"ot_{split_name}_{ct}")
        row_ctrl = compute_metrics(pred_ctrl[ct_mask], true_pca[ct_mask], pred_ctrl[ct_mask],
                                   label=f"ctrl_{split_name}_{ct}")
        for row, method in [(row_ot,"ot_pilot"), (row_ctrl,"ctrl_baseline")]:
            row["split"] = split_name; row["cell_type"] = ct; row["method"] = method
            per_cell_rows.append(row)

per_cell_df = pd.DataFrame(per_cell_rows)
print("=== Per-Cell-Type Evaluation ===")
print(per_cell_df[["split","cell_type","method","n_cells","r2_top50","r2_top20","pearson_row"]].to_string(index=False))
per_cell_df.to_csv(os.path.join(cfg.results_dir, "eval_per_cell_nb12.csv"), index=False)
print("\nSaved: eval_per_cell_nb12.csv")

=== Per-Cell-Type Evaluation ===
split cell_type        method  n_cells  r2_top50  r2_top20  pearson_row
 test      A549      ot_pilot    12626  0.707683  0.635917     0.859504
 test      A549 ctrl_baseline    12626  0.679077  0.604823     0.845225
 test      K562      ot_pilot    12522  0.584228  0.521069     0.802174
 test      K562 ctrl_baseline    12522  0.561737  0.496689     0.793886
 test      MCF7      ot_pilot    26095  0.482382  0.391760     0.729059
 test      MCF7 ctrl_baseline    26095  0.440954  0.347849     0.684158
  ood      A549      ot_pilot    11936  0.703901  0.632839     0.858646
  ood      A549 ctrl_baseline    11936  0.695625  0.623705     0.855418
  ood      K562      ot_pilot    13062  0.577491  0.513176     0.799145
  ood      K562 ctrl_baseline    13062  0.569718  0.504820     0.797399
  ood      MCF7      ot_pilot    24868  0.481595  0.386387     0.720234
  ood      MCF7 ctrl_baseline    24868  0.469776  0.373042     0.710530

Saved: eval_per_cell_nb12.csv


## 11) Per-supergroup evaluation

In [20]:
per_sg_rows = []
for split_name, split_mask_arr, pred_ot, pred_ctrl, true_pca in [
    ("test", is_test, pred_ot_test, pred_ctrl_test, true_test),
    ("ood",  is_ood,  pred_ot_ood,  pred_ctrl_ood,  true_ood),
]:
    dv = cond_vals[split_mask_arr]
    drugs_in_split = sorted(set(dv))
    for sg_id in range(n_sg):
        sg_drugs = [d for d in drugs_in_split if all_drug_sg.get(d) == sg_id]
        if not sg_drugs:
            continue
        sg_mask = np.isin(dv, sg_drugs)
        if sg_mask.sum() == 0:
            continue
        for pred, method in [(pred_ot, "ot_pilot"), (pred_ctrl, "ctrl_baseline")]:
            row = compute_metrics(pred[sg_mask], true_pca[sg_mask], pred_ctrl[sg_mask],
                                  label=f"{method}_sg{sg_id}_{split_name}")
            row.update({"split": split_name, "supergroup": sg_id,
                        "n_drugs": len(sg_drugs), "method": method})
            per_sg_rows.append(row)

per_sg_df = pd.DataFrame(per_sg_rows)
print("=== Per-Supergroup Evaluation ===")
print(per_sg_df[["split","supergroup","method","n_cells","n_drugs","r2_top50","pearson_row"]].to_string(index=False))
per_sg_df.to_csv(os.path.join(cfg.results_dir, "eval_per_sg_nb12.csv"), index=False)
print("\nSaved: eval_per_sg_nb12.csv")

=== Per-Supergroup Evaluation ===
split  supergroup        method  n_cells  n_drugs  r2_top50  pearson_row
 test           0      ot_pilot    26044       13  0.570971     0.779621
 test           0 ctrl_baseline    26044       13  0.570052     0.779598
 test           1      ot_pilot     3092        2  0.481291     0.778232
 test           1 ctrl_baseline     3092        2 -0.052370     0.316820
 test           2      ot_pilot     7759        5  0.542103     0.765815
 test           2 ctrl_baseline     7759        5  0.536503     0.762475
 test           5      ot_pilot    14348        8  0.576663     0.785407
 test           5 ctrl_baseline    14348        8  0.576215     0.785228
  ood           0      ot_pilot    25265       13  0.567744     0.778271
  ood           0 ctrl_baseline    25265       13  0.566263     0.778014
  ood           2      ot_pilot     5911        3  0.547618     0.768591
  ood           2 ctrl_baseline     5911        3  0.546091     0.767707
  ood           3

## 12) Optional — compare against NB11 residual predictions

Loads NB11's saved test predictions if available. Fails gracefully if not present.

In [21]:
nb11_pred_path = os.path.join(cfg.nb11_res_dir, "test_preds_residual_only.npy")
nb11_true_path = os.path.join(cfg.nb11_res_dir, "test_true_residual_only.npy")
nb11_meta_path = os.path.join(cfg.nb11_res_dir, "test_meta_residual_only.csv")

if all(os.path.exists(p) for p in [nb11_pred_path, nb11_true_path, nb11_meta_path]):
    nb11_pred = np.load(nb11_pred_path).astype(np.float32)
    nb11_true = np.load(nb11_true_path).astype(np.float32)
    nb11_meta = pd.read_csv(nb11_meta_path)
    nb11_ctrl = np.vstack([get_ctrl_anchor(ct) for ct in nb11_meta["cell_type"].astype(str)])
    row_nb11  = compute_metrics(nb11_pred, nb11_true, nb11_ctrl, label="nb11_residual_test")
    print("=== Three-way comparison (test split) ===")
    cmp = pd.DataFrame([
        overall_df[overall_df.label == "ctrl_baseline_test"].iloc[0],
        pd.Series(row_nb11),
        overall_df[overall_df.label == "ot_test"].iloc[0],
    ])
    print(cmp[["label","n_cells","r2_top50","r2_top20","pearson_row"]].to_string(index=False))
else:
    print(f"NB11 saved predictions not found (expected: {nb11_pred_path}).")
    print("Showing NB12 OT vs ctrl baseline only:")
    cmp = overall_df[overall_df["label"].isin(["ot_test","ctrl_baseline_test","ot_ood","ctrl_baseline_ood"])]
    print(cmp[["label","n_cells","r2_top50","r2_top20","pearson_row"]].to_string(index=False))

NB11 saved predictions not found (expected: /content/drive/MyDrive/ChemDFM/results_nb11_scaffold/test_preds_residual_only.npy).
Showing NB12 OT vs ctrl baseline only:
             label  n_cells  r2_top50  r2_top20  pearson_row
           ot_test    51243  0.562783  0.483518     0.779067
ctrl_baseline_test    51243  0.529141  0.447537     0.750658
            ot_ood    49866  0.559926  0.478590     0.774035
 ctrl_baseline_ood    49866  0.550014  0.467559     0.767965


## 13) Save run summary JSON

In [22]:
def py_native(x):
    if isinstance(x, (np.bool_,)):   return bool(x)
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.floating,)): return float(x)
    if isinstance(x, dict):          return {k: py_native(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [py_native(v) for v in x]
    return x

summary = {
    "notebook":  "NB12_Mini_DrugConditioned_OT_Pilot",
    "method":    "scaffold_cluster_conditioned_gaussian_ot",
    "config":    py_native(asdict(cfg)),
    "n_supergroups_fitted": int(n_sg),
    "n_train_drugs": len(drug_centroids),
    "ot_maps_fitted_pct": round(float(fit_log_df["fitted"].mean() * 100), 2),
    "overall_results": py_native(
        overall_df[["label","n_cells","r2_full","r2_top20","r2_top50","pearson_row"]].to_dict("records")
    ),
    "per_cell_results": py_native(
        per_cell_df[["split","cell_type","method","n_cells","r2_top50"]].to_dict("records")
    ),
    "artifacts": {
        "ot_maps":              f"{cfg.art_dir}/ot_maps_nb12.pkl",
        "fit_log":              f"{cfg.art_dir}/ot_map_fit_log.csv",
        "sg_assignments_train": f"{cfg.art_dir}/train_drug_supergroup_assignments.csv",
        "routing_ood_test":     f"{cfg.art_dir}/ood_test_drug_routing.csv",
        "eval_overall":         f"{cfg.results_dir}/eval_overall_nb12.csv",
        "eval_per_cell":        f"{cfg.results_dir}/eval_per_cell_nb12.csv",
        "eval_per_sg":          f"{cfg.results_dir}/eval_per_sg_nb12.csv",
        "run_summary":          f"{cfg.results_dir}/run_summary_nb12.json",
    },
    "limitations": [
        "Monge-Gaussian maps are linear and cannot capture multimodal drug effects.",
        "Routing unseen drugs by centroid proximity is approximate; structurally isolated OOD drugs may be poorly served.",
        "Group-level maps average over all member drugs; drug-specific dose response is suppressed.",
        "PCA truncation (50 dims) loses high-frequency gene expression variability.",
        "This is a pilot, not a final generalisation benchmark.",
    ]
}
with open(os.path.join(cfg.results_dir, "run_summary_nb12.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("Saved: run_summary_nb12.json")
print("\n=== NB12 Complete ===")
print(f"  Supergroups: {n_sg}  |  Maps fitted: {fit_log_df['fitted'].sum()}/{len(fit_log_df)}")
for r in summary["overall_results"]:
    print(f"  {r['label']:35s} r2_top50={r['r2_top50']:.4f}  pearson={r['pearson_row']:.4f}")

Saved: run_summary_nb12.json

=== NB12 Complete ===
  Supergroups: 8  |  Maps fitted: 24/24
  ot_test                             r2_top50=0.5628  pearson=0.7791
  ctrl_baseline_test                  r2_top50=0.5291  pearson=0.7507
  ot_ood                              r2_top50=0.5599  pearson=0.7740
  ctrl_baseline_ood                   r2_top50=0.5500  pearson=0.7680


## Summary and what NB13 should consume

### What NB12 demonstrated

- Scaffold-cluster-conditioned Gaussian OT is a minimal, leakage-free pilot for unseen-drug-aware transport.
- Routing test/OOD drugs by structural proximity introduces no phenotypic leakage.
- Group-level Monge maps transfer to unseen drugs by structural proxy.

### Honest limitations

1. **Linearity**: Monge maps between Gaussians are linear — non-Gaussian or multimodal drug effects are not captured.
2. **Approximate routing**: Structurally novel OOD drugs may be far from all training supergroups (see `dist_to_sg_center` in routing CSV).
3. **Group averaging**: All drugs in a supergroup share the same map; dose-specific or drug-specific perturbations are averaged out.
4. **Not competitive with NB11 on seen drugs**: The residual model (NB11) uses optimised drug embeddings — this OT pilot is a geometric baseline for the *unseen-drug* setting.

### What NB13 should consume

| Artifact | Purpose in NB13 |
|----------|-----------------|
| `ot_maps_nb12.pkl` | Initialise or regularise a flow/conditional OT model |
| `ood_test_drug_routing.csv` | Routing quality analysis; identify structurally isolated OOD drugs |
| `train_drug_supergroup_assignments.csv` | Supergroup labels for conditioning a learned map |
| `eval_per_sg_nb12.csv` | Identify supergroups that benefit most from richer transport |
| `run_summary_nb12.json` | Config reproducibility |

**NB13 direction**: Replace the Gaussian map with a conditional flow or shallow MLP transport network conditioned on Morgan fingerprints. The supergroup assignments from NB12 provide structural grouping as a prior. The routing logic can be retained or replaced with learned fingerprint-based routing.
